## Pod Security Admission, etcd encryption & image security

### Pod Security Admission

A per-pod security context is great; you also want a **blanket policy** that rejects non-compliant pods. **PSA** does that (replacing the removed `PodSecurityPolicy`). Three **levels** — **`privileged`** (anything), **`baseline`** (no privileged/hostPath/host namespaces), **`restricted`** (non-root, drop all caps, seccomp) — applied per-namespace by label, in three **modes**: **`enforce`** (reject), **`audit`** (log), **`warn`** (message).

```yaml
kind: Namespace
metadata:
  name: secure
  labels:
    pod-security.kubernetes.io/enforce: restricted
    pod-security.kubernetes.io/warn: restricted
```

Migration recipe: start with `warn`+`audit` at the target level, fix the pods, then switch to `enforce`. For finer rules, **Gatekeeper** (Rego) or **Kyverno** (YAML) via admission webhooks — beyond CKA.

### etcd encryption at rest

Secret data in `etcd` is base64, **not encrypted** by default — a data-dir or backup attacker reads every Secret. Fix with an **`EncryptionConfiguration`** the API server reads (`--encryption-provider-config`), encrypting `secrets` (and `configmaps`) with an `aescbc` or `kms` provider before writing. First provider encrypts; all decrypt — so rotate by adding the new key first, re-encrypting (`kubectl get secrets -A -o json | kubectl replace -f -`), then dropping the old. Keep `identity: {}` last during migration. It protects **at rest only** — not in transit (TLS already) nor in the pod.

### Image security

Pin **digests** (`nginx@sha256:...`), not moving tags. Private registries via `imagePullSecret` (notebook 05). Sign with **cosign/sigstore**, verify at admission (Kyverno). Scan in CI (**Trivy**). Runtime detection with **Falco**. On our map this is the **PSA** chip in the Security box — the namespace-wide enforcement over everything module 10 hardens.